# LeetCode 3019 — Number of Changing Keys bằng Python

Link đề: https://leetcode.com/problems/number-of-changing-keys/ · Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def countKeyChanges(self, s: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm **số lần người dùng phải đổi sang một phím khác** khi gõ chuỗi `s`, trong đó **chữ hoa và chữ thường của cùng một chữ cái dùng chung một phím** (dùng Shift/Caps Lock **không** tính là đổi phím).
2. Biết **chuẩn hóa chữ hoa/thường** bằng `lower()` và so sánh **cặp ký tự liền kề** `s[i - 1]`, `s[i]`.
3. Biết ba cách duyệt cặp liền kề: **chỉ số `range(1, n)`**, **`zip(t, t[1:])`**, và **`itertools.groupby`** (đếm số "nhóm" liên tiếp trừ `1`).
4. Đi từ cách **so sánh từng cặp có `lower()`** → **chuẩn hóa cả chuỗi rồi đếm cặp khác nhau** → **`groupby`** → **mẹo bit `ord(c) & 31`**.
5. Tránh các lỗi kinh điển: quên bỏ qua chữ hoa/thường, bắt đầu vòng lặp từ `0` (so sánh `s[0]` với `s[-1]`), đếm **số nhóm** thay vì **số lần đổi** (lệch `1`), và đếm **số chữ cái khác nhau** thay vì số lần đổi phím.

## 2. Đề bài & ví dụ

Bạn được cho một chuỗi `s` (đánh chỉ số từ `0`) mô tả **các ký tự mà người dùng đã gõ**. Việc **đổi phím** được định nghĩa là: **dùng một phím khác** với phím vừa dùng ngay trước đó. Ví dụ gõ `s = "ab"` có một lần đổi phím, nhưng gõ `s = "bBBb"` thì **không** có lần đổi phím nào.

**Lưu ý:** các phím điều khiển như **Shift** hoặc **Caps Lock** không được tính là một lần đổi phím: chữ hoa `'A'` và chữ thường `'a'` được coi là **cùng một phím**.

Hãy trả về **số lần người dùng đã đổi phím**.

**Ví dụ 1**

- Input: `s = "aAbBcC"`
- Output: `2`
- Giải thích: `'a'` sang `'A'`: không đổi phím (cùng phím `a`). `'A'` sang `'b'`: đổi phím (lần 1). `'b'` sang `'B'`: không đổi. `'B'` sang `'c'`: đổi phím (lần 2). `'c'` sang `'C'`: không đổi. Tổng `2`.

**Ví dụ 2**

- Input: `s = "AaAaAaaA"`
- Output: `0`
- Giải thích: toàn bộ chuỗi chỉ dùng phím `a` (chỉ khác chữ hoa/thường), không có lần đổi phím nào.

**Ràng buộc:** `1 <= s.length <= 100`; `s` chỉ gồm các chữ cái tiếng Anh in hoa hoặc in thường.

In [1]:
from itertools import groupby
import random

examples = [
    ("aAbBcC", 2),
    ("AaAaAaaA", 0),
    ("ab", 1),
    ("bBBb", 0),
    ("a", 0),
]
for s, expected in examples:
    print("s =", repr(s), "-> kỳ vọng", expected)

s = 'aAbBcC' -> kỳ vọng 2
s = 'AaAaAaaA' -> kỳ vọng 0
s = 'ab' -> kỳ vọng 1
s = 'bBBb' -> kỳ vọng 0
s = 'a' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Chuỗi là dãy ký tự, truy cập bằng chỉ số

`s[i]` là ký tự thứ `i` (đếm từ `0`), `len(s)` là độ dài. Chỉ số **âm** đếm từ cuối: `s[-1]` là ký tự cuối. Ta sẽ so sánh ký tự `s[i]` với ký tự đứng ngay trước nó `s[i - 1]`.

In [2]:
s = "aAbBcC"
print(len(s), s[0], s[1], s[-1])           # 6 a A C
for i in range(1, len(s)):
    print(f"i={i}: trước={s[i - 1]!r}, hiện tại={s[i]!r}")

6 a A C
i=1: trước='a', hiện tại='A'
i=2: trước='A', hiện tại='b'
i=3: trước='b', hiện tại='B'
i=4: trước='B', hiện tại='c'
i=5: trước='c', hiện tại='C'


### 3.2. Chữ hoa / chữ thường: `lower()`, `upper()`, `casefold()`

`'a' == 'A'` là **`False`** trong Python (phân biệt hoa/thường). Hàm `s.lower()` đổi toàn bộ về chữ thường; `s.upper()` đổi về chữ hoa. Để coi `'a'` và `'A'` là cùng một phím, ta **chuẩn hóa** cả hai về cùng một dạng rồi mới so sánh.

In [3]:
print('a' == 'A')                           # False
print('a'.lower() == 'A'.lower())           # True
print("aAbBcC".lower(), "aAbBcC".upper())   # aabbcc AABBCC
print("Hello".swapcase())                   # hELLO: đổi hoa <-> thường

False
True
aabbcc AABBCC
hELLO


### 3.3. Hai ký tự liền kề: `zip(t, t[1:])`

`t[1:]` là chuỗi bỏ ký tự đầu. `zip(t, t[1:])` ghép **ký tự thứ `i` với ký tự thứ `i + 1`**, đúng bằng các cặp liền kề, và tự dừng khi một chuỗi hết (nên không bị lỗi chỉ số). Có `len(t) - 1` cặp.

In [4]:
t = "aabbcc"
print(list(zip(t, t[1:])))                  # [('a','a'), ('a','b'), ('b','b'), ('b','c'), ('c','c')]
print(sum(a != b for a, b in zip(t, t[1:])))   # 2: số cặp khác nhau

[('a', 'a'), ('a', 'b'), ('b', 'b'), ('b', 'c'), ('c', 'c')]
2


### 3.4. `itertools.groupby`: gom các phần tử **liên tiếp** giống nhau

`groupby(t)` chia dãy thành các **nhóm liên tiếp** có cùng giá trị: `"aabbbc"` thành nhóm `a`, `b`, `c`. Chú ý: chỉ gom những phần tử **liền kề**, không gom các phần tử giống nhau nằm xa nhau (`"aba"` thành `3` nhóm).

In [5]:
for key, group in groupby("aabbbca"):
    print(key, len(list(group)))             # a 2, b 3, c 1, a 1
print(len(list(groupby("aba"))))              # 3 nhóm

a 2
b 3
c 1
a 1
3


### 3.5. Đếm số lần đổi = số nhóm trừ `1`

Nếu chuỗi (đã chuẩn hóa) chia thành `g` nhóm liên tiếp, thì ranh giới giữa các nhóm có đúng `g - 1` chỗ, mỗi ranh giới là **một lần đổi**. Ví dụ `"aabbcc"` có `3` nhóm nên `2` lần đổi.

In [6]:
t = "aabbcc"
g = sum(1 for _ in groupby(t))
print("số nhóm:", g, "-> số lần đổi:", g - 1)   # 3 -> 2

số nhóm: 3 -> số lần đổi: 2


### 3.6. Mẹo bit: `ord(c) & 31` cho cùng giá trị với chữ hoa và chữ thường

Mã ASCII của `'A'` là `65 = 0b1000001` và của `'a'` là `97 = 0b1100001`: chúng chỉ **khác bit thứ 5** (giá trị `32`). Phép `& 31` giữ lại `5` bit thấp nên `'A'` và `'a'` cùng cho `1`, `'B'` và `'b'` cùng cho `2`, ... Đây là cách "chuẩn hóa" không cần `lower()` (chỉ đúng cho chữ cái ASCII).

In [7]:
for c in "aAzZ":
    print(c, ord(c), bin(ord(c)), ord(c) & 31)
print(ord("a") & 31 == ord("A") & 31)           # True

a 97 0b1100001 1
A 65 0b1000001 1
z 122 0b1111010 26
Z 90 0b1011010 26
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy "gõ" từng ký tự và so với phím ngay trước đó **không phân biệt hoa/thường**. Với `s = "aAbBcC"`:

| `i` | `s[i - 1]` | `s[i]` | cùng phím? (so sánh khi đã `lower`) | đổi phím? | số lần đổi |
|---|---|---|---|---|---|
| 1 | `a` | `A` | `a` và `a` | không | 0 |
| 2 | `A` | `b` | `a` và `b` | **có** | 1 |
| 3 | `b` | `B` | `b` và `b` | không | 1 |
| 4 | `B` | `c` | `b` và `c` | **có** | 2 |
| 5 | `c` | `C` | `c` và `c` | không | 2 |

Chỉ có `n - 1` cặp liền kề (ký tự đầu không có "phím trước"), và ta đếm các cặp thuộc **hai phím khác nhau**. Không cần nhớ gì hơn ngoài **phím của ký tự ngay trước**.

In [8]:
s = "aAbBcC"
changes = 0
for i in range(1, len(s)):
    a, b = s[i - 1].lower(), s[i].lower()
    different = a != b
    changes += different
    print(f"i={i}: {s[i - 1]!r} -> {s[i]!r} | {a!r} vs {b!r} | đổi phím: {different!s:5} | tổng={changes}")

i=1: 'a' -> 'A' | 'a' vs 'a' | đổi phím: False | tổng=0
i=2: 'A' -> 'b' | 'a' vs 'b' | đổi phím: True  | tổng=1
i=3: 'b' -> 'B' | 'b' vs 'b' | đổi phím: False | tổng=1
i=4: 'B' -> 'c' | 'b' vs 'c' | đổi phím: True  | tổng=2
i=5: 'c' -> 'C' | 'c' vs 'c' | đổi phím: False | tổng=2


## 5. Cách 1 — Brute Force / baseline: so sánh từng cặp liền kề, `lower()` hai phía

**Ý tưởng.** Duyệt `i` từ `1` đến `n - 1`; nếu `s[i].lower() != s[i - 1].lower()` thì `count += 1`. Làm đúng định nghĩa.

In [9]:
def count_key_changes_pairs(s: str) -> int:
    count = 0
    for i in range(1, len(s)):                         # từ ký tự thứ hai trở đi
        if s[i].lower() != s[i - 1].lower():           # hai phím khác nhau (không phân biệt hoa/thường)
            count += 1
    return count


for s, expected in examples:
    print(repr(s), "->", count_key_changes_pairs(s), "(kỳ vọng", expected, ")")

'aAbBcC' -> 2 (kỳ vọng 2 )
'AaAaAaaA' -> 0 (kỳ vọng 0 )
'ab' -> 1 (kỳ vọng 1 )
'bBBb' -> 0 (kỳ vọng 0 )
'a' -> 0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Một lượt qua `n - 1` cặp, mỗi cặp `O(1)`: **Time `O(n)`**, **Space `O(1)`**.

**Nút thắt.** Đã tối ưu về bậc. Có thể viết ngắn hơn, hoặc tránh gọi `lower()` hai lần mỗi cặp bằng cách chuẩn hóa chuỗi **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

Không có thay đổi về bậc, chỉ có **cách nhìn gọn hơn**:

1. **Chuẩn hóa một lần:** `t = s.lower()`. Bài toán thành "đếm số vị trí `i` mà `t[i] != t[i - 1]`".
2. **Đếm cặp khác nhau** bằng `sum(a != b for a, b in zip(t, t[1:]))`, hoặc
3. **Đếm nhóm liên tiếp:** số lần đổi bằng **số nhóm trừ `1`**.
4. Hoặc **dùng một biến `prev`** để nhớ phím trước và tránh tạo lát cắt `t[1:]` (bộ nhớ `O(1)`).

## 7. Các cách tiếp theo

### Cách 2 — Chuẩn hóa cả chuỗi rồi `zip` các cặp liền kề

**Giải thích.** `t = s.lower()`; `zip(t, t[1:])` cho các cặp liền kề; `sum(a != b ...)` đếm cặp khác nhau (`True` cộng như `1`).

In [10]:
def count_key_changes_zip(s: str) -> int:
    t = s.lower()                                      # chuẩn hóa chữ hoa/thường một lần
    return sum(a != b for a, b in zip(t, t[1:]))       # đếm các cặp liền kề khác nhau


for s, expected in examples:
    print(repr(s), "->", count_key_changes_zip(s), "(kỳ vọng", expected, ")")

'aAbBcC' -> 2 (kỳ vọng 2 )
'AaAaAaaA' -> 0 (kỳ vọng 0 )
'ab' -> 1 (kỳ vọng 1 )
'bBBb' -> 0 (kỳ vọng 0 )
'a' -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (chuỗi `t` và lát cắt `t[1:]`).

### Cách 3 — `itertools.groupby`: số nhóm trừ `1`

**Giải thích.** `groupby(s.lower())` chia chuỗi thành các **nhóm liên tiếp** cùng phím. Số lần đổi phím là `số nhóm - 1`.

In [11]:
def count_key_changes_groupby(s: str) -> int:
    groups = sum(1 for _ in groupby(s.lower()))        # số đoạn liên tiếp cùng phím
    return groups - 1                                  # n đoạn có n - 1 chỗ nối


for s, expected in examples:
    print(repr(s), "->", count_key_changes_groupby(s), "(kỳ vọng", expected, ")")

# Quan sát các nhóm của ví dụ 1
print([(k, len(list(g))) for k, g in groupby("aAbBcC".lower())])

'aAbBcC' -> 2 (kỳ vọng 2 )
'AaAaAaaA' -> 0 (kỳ vọng 0 )
'ab' -> 1 (kỳ vọng 1 )
'bBBb' -> 0 (kỳ vọng 0 )
'a' -> 0 (kỳ vọng 0 )
[('a', 2), ('b', 2), ('c', 2)]


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (`groupby` lười, ta chỉ đếm).

### Cách 4 — Một biến `prev` và mẹo bit `ord(c) & 31`

**Giải thích.** Không tạo chuỗi mới: nhớ **mã phím** của ký tự trước bằng `prev = ord(ch) & 31` (cùng giá trị cho chữ hoa và thường, mục 3.6). Mỗi ký tự mới: tính `cur = ord(ch) & 31`; nếu `cur != prev` thì đếm; rồi `prev = cur`. Chỉ cần `O(1)` bộ nhớ.

In [12]:
def count_key_changes_prev(s: str) -> int:
    count = 0
    prev = ord(s[0]) & 31                              # mã phím của ký tự đầu (chữ cái ASCII)
    for ch in s[1:]:
        cur = ord(ch) & 31
        if cur != prev:
            count += 1
            prev = cur                                 # chỉ cần cập nhật khi đổi (nếu bằng nhau thì giữ nguyên)
    return count


for s, expected in examples:
    print(repr(s), "->", count_key_changes_prev(s), "(kỳ vọng", expected, ")")

'aAbBcC' -> 2 (kỳ vọng 2 )
'AaAaAaaA' -> 0 (kỳ vọng 0 )
'ab' -> 1 (kỳ vọng 1 )
'bBBb' -> 0 (kỳ vọng 0 )
'a' -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (ngoài lát `s[1:]`; có thể dùng chỉ số để tránh).

In [13]:
# Code nộp LeetCode cho Cách 1 (rõ ràng nhất)
class Solution:
    def countKeyChanges(self, s: str) -> int:
        count = 0
        for i in range(1, len(s)):
            if s[i].lower() != s[i - 1].lower():
                count += 1
        return count

## 8. TRACE từng bước

Ta theo dõi (a) Cách 1 với `"aAbBBcAa"` và (b) Cách 3 qua các nhóm.

In [14]:
def trace_pairs(s):
    count = 0
    print("s =", repr(s))
    for i in range(1, len(s)):
        p, c = s[i - 1], s[i]
        diff = p.lower() != c.lower()
        if diff:
            count += 1
        print(f"i={i}: {p!r} -> {c!r} | {p.lower()!r} vs {c.lower()!r} | {'ĐỔI PHÍM' if diff else 'cùng phím'} | count={count}")
    print("=> kết quả:", count)
    return count


assert trace_pairs("aAbBBcAa") == 3
assert trace_pairs("aAbBcC") == 2

s = 'aAbBBcAa'
i=1: 'a' -> 'A' | 'a' vs 'a' | cùng phím | count=0
i=2: 'A' -> 'b' | 'a' vs 'b' | ĐỔI PHÍM | count=1
i=3: 'b' -> 'B' | 'b' vs 'b' | cùng phím | count=1
i=4: 'B' -> 'B' | 'b' vs 'b' | cùng phím | count=1
i=5: 'B' -> 'c' | 'b' vs 'c' | ĐỔI PHÍM | count=2
i=6: 'c' -> 'A' | 'c' vs 'a' | ĐỔI PHÍM | count=3
i=7: 'A' -> 'a' | 'a' vs 'a' | cùng phím | count=3
=> kết quả: 3
s = 'aAbBcC'
i=1: 'a' -> 'A' | 'a' vs 'a' | cùng phím | count=0
i=2: 'A' -> 'b' | 'a' vs 'b' | ĐỔI PHÍM | count=1
i=3: 'b' -> 'B' | 'b' vs 'b' | cùng phím | count=1
i=4: 'B' -> 'c' | 'b' vs 'c' | ĐỔI PHÍM | count=2
i=5: 'c' -> 'C' | 'c' vs 'c' | cùng phím | count=2
=> kết quả: 2


In [15]:
def trace_groups(s):
    groups = [(k, len(list(g))) for k, g in groupby(s.lower())]
    print("s.lower() =", repr(s.lower()))
    print("các nhóm:", groups)
    print(f"số nhóm = {len(groups)} -> số lần đổi = {len(groups) - 1}")
    return len(groups) - 1


assert trace_groups("aAbBBcAa") == 3
assert trace_groups("AaAaAaaA") == 0

s.lower() = 'aabbbcaa'
các nhóm: [('a', 2), ('b', 3), ('c', 1), ('a', 2)]
số nhóm = 4 -> số lần đổi = 3
s.lower() = 'aaaaaaaa'
các nhóm: [('a', 8)]
số nhóm = 1 -> số lần đổi = 0


Bảng tóm tắt (Cách 1, `"aAbBBcAa"`):

| `i` | cặp | cùng phím? | `count` |
|---|---|---|---|
| 1 | `a, A` | có | 0 |
| 2 | `A, b` | không | 1 |
| 3 | `b, B` | có | 1 |
| 4 | `B, B` | có | 1 |
| 5 | `B, c` | không | 2 |
| 6 | `c, A` | không | 3 |
| 7 | `A, a` | có | 3 |

Các nhóm (sau `lower()`): `aa`, `bbb`, `c`, `aa` là `4` nhóm, nên số lần đổi là `4 - 1 = 3`, khớp với bảng.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 1 (so sánh từng cặp liền kề). Đúng class `Solution`, đúng method `countKeyChanges(self, s: str) -> int`.

In [16]:
class Solution:
    def countKeyChanges(self, s: str) -> int:
        count = 0
        for i in range(1, len(s)):                          # từ ký tự thứ hai
            if s[i].lower() != s[i - 1].lower():            # khác phím (bỏ qua hoa/thường)
                count += 1
        return count


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.countKeyChanges(s))

'aAbBcC' -> 2
'AaAaAaaA' -> 0
'ab' -> 1
'bBBb' -> 0
'a' -> 0


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa.** Một lần đổi phím xảy ra khi phím của ký tự hiện tại **khác** phím của ký tự ngay trước. "Phím" của một chữ cái là chữ cái đó **bỏ phân biệt hoa/thường**, tức `ch.lower()`. Vậy lần đổi ⇔ `s[i].lower() != s[i - 1].lower()`.
- **Mỗi cặp đúng một lần.** Các lần đổi phím chỉ có thể xảy ra giữa hai ký tự **liền kề**; ta xét mọi cặp `(i - 1, i)` với `i = 1..n - 1` đúng một lần và không có cặp nào khác, nên không đếm thiếu hay thừa.
- **Ký tự đầu.** Ký tự đầu tiên không có "phím trước", nên không tạo lần đổi: vòng lặp bắt đầu từ `i = 1`.
- **Tương đương `groupby`.** Chuỗi chuẩn hóa chia thành `g` nhóm liên tiếp; các chỗ nối giữa nhóm chính là các vị trí `t[i] != t[i - 1]`, đúng `g - 1` chỗ.
- **Mẹo bit.** `ord(c) & 31` giống nhau cho `'a'` và `'A'` (và với mọi cặp hoa/thường), và khác nhau giữa hai chữ cái khác nhau, nên tương đương `lower()` trên các chữ cái ASCII.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

- **Cách 1 (so sánh cặp, `lower()` hai phía):** Time `O(n)`, Space `O(1)`.
- **Cách 2 (`lower` cả chuỗi + `zip`):** Time `O(n)`, Space `O(n)`.
- **Cách 3 (`groupby`):** Time `O(n)`, Space `O(1)` (không tính chuỗi `lower`).
- **Cách 4 (`prev` + `ord & 31`):** Time `O(n)`, Space `O(1)`.
- Phải xem mọi ký tự nên `O(n)` là tối ưu. Ở đây `n <= 100`.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "pairs": count_key_changes_pairs,
    "zip": count_key_changes_zip,
    "groupby": count_key_changes_groupby,
    "prev": count_key_changes_prev,
    "submit": sol.countKeyChanges,
}

for s, expected in examples:
    for key, fn in impls.items():
        got = fn(s)
        assert got == expected, (key, s, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(s, expected, label):
    for key, fn in impls.items():
        got = fn(s)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all("a", 0, "một ký tự: không có cặp nào")
check_all("A", 0, "một chữ hoa")
check_all("aa", 0, "hai ký tự giống hệt")
check_all("aA", 0, "hoa/thường của cùng phím: không đổi")
check_all("ab", 1, "hai phím khác nhau")
check_all("aB", 1, "khác phím, khác hoa/thường")
check_all("abab", 3, "xen kẽ hai phím: đổi ở mọi cặp")
check_all("aAaA", 0, "toàn phím a, hoa/thường xen kẽ")
check_all("AbCdEf", 5, "mọi cặp đều khác phím")
check_all("aabbcc", 2, "ba cặp chữ lặp")
check_all("aAbBbBaA", 2, "a -> b -> a: hai lần đổi")
check_all("z" * 100, 0, "n = 100 cùng một chữ")
check_all("aB" * 50, 99, "n = 100 xen kẽ a, B: mọi cặp đổi phím")
check_all("zZ" * 50, 0, "n = 100 chỉ phím z")

PASS một ký tự: không có cặp nào -> 0
PASS một chữ hoa -> 0
PASS hai ký tự giống hệt -> 0
PASS hoa/thường của cùng phím: không đổi -> 0
PASS hai phím khác nhau -> 1
PASS khác phím, khác hoa/thường -> 1
PASS xen kẽ hai phím: đổi ở mọi cặp -> 3
PASS toàn phím a, hoa/thường xen kẽ -> 0
PASS mọi cặp đều khác phím -> 5
PASS ba cặp chữ lặp -> 2
PASS a -> b -> a: hai lần đổi -> 2
PASS n = 100 cùng một chữ -> 0
PASS n = 100 xen kẽ a, B: mọi cặp đổi phím -> 99
PASS n = 100 chỉ phím z -> 0


### 12.2. Tính chất và phản ví dụ

In [19]:
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def random_case(word, rng):
    return "".join(c.upper() if rng.random() < 0.5 else c for c in word)


rng = random.Random(11)
# Tính chất 1: đổi hoa/thường tùy ý không làm đổi kết quả
for _ in range(500):
    base = "".join(rng.choice(LETTERS[:4]) for _ in range(rng.randint(1, 100)))
    assert sol.countKeyChanges(random_case(base, rng)) == sol.countKeyChanges(base)
    assert sol.countKeyChanges(base.swapcase()) == sol.countKeyChanges(base)
print("PASS đổi hoa/thường tùy ý không đổi đáp án")

# Tính chất 2: 0 <= đáp án <= n - 1; đáp án 0 <=> mọi ký tự cùng một chữ cái
for _ in range(500):
    s = random_case("".join(rng.choice(LETTERS[:3]) for _ in range(rng.randint(1, 100))), rng)
    r = sol.countKeyChanges(s)
    assert 0 <= r <= len(s) - 1
    assert (r == 0) == (len(set(s.lower())) == 1)
print("PASS 0 <= đáp án <= n - 1; bằng 0 <=> chỉ một chữ cái")

# Tính chất 3: đáp án = số nhóm liên tiếp - 1
for _ in range(500):
    s = "".join(rng.choice(LETTERS[:3]) for _ in range(rng.randint(1, 100)))
    assert sol.countKeyChanges(s) == len(list(groupby(s))) - 1
print("PASS đáp án = số nhóm - 1")

# Tính chất 4: đảo ngược chuỗi không đổi đáp án (đối xứng)
for _ in range(500):
    s = random_case("".join(rng.choice(LETTERS[:4]) for _ in range(rng.randint(1, 100))), rng)
    assert sol.countKeyChanges(s) == sol.countKeyChanges(s[::-1])
print("PASS đảo ngược chuỗi không đổi đáp án")

# Phản ví dụ 1: phân biệt hoa/thường
def sai_phan_biet(s):
    return sum(1 for i in range(1, len(s)) if s[i] != s[i - 1])

assert sai_phan_biet("aAbBcC") == 5 != 2
print("PASS phản ví dụ: so sánh trực tiếp cho", sai_phan_biet("aAbBcC"), "thay vì 2")

# Phản ví dụ 2: đếm số chữ cái khác nhau
def sai_chu_cai_khac_nhau(s):
    return len(set(s.lower())) - 1

assert sai_chu_cai_khac_nhau("abab") == 1 and sol.countKeyChanges("abab") == 3
print("PASS phản ví dụ: len(set) - 1 cho 1 thay vì 3 ở 'abab'")

PASS đổi hoa/thường tùy ý không đổi đáp án
PASS 0 <= đáp án <= n - 1; bằng 0 <=> chỉ một chữ cái
PASS đáp án = số nhóm - 1
PASS đảo ngược chuỗi không đổi đáp án
PASS phản ví dụ: so sánh trực tiếp cho 5 thay vì 2
PASS phản ví dụ: len(set) - 1 cho 1 thay vì 3 ở 'abab'


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
rng = random.Random(3019)
for trial in range(3000):
    n = rng.randint(1, 100)
    alphabet = rng.choice([LETTERS[:2], LETTERS[:5], LETTERS])
    s = random_case("".join(rng.choice(alphabet) for _ in range(n)), rng)
    # mô phỏng độc lập: trạng thái "phím hiện tại"
    cur, changes = None, 0
    for ch in s:
        key = ch.lower()
        if cur is not None and key != cur:
            changes += 1
        cur = key
    for name, fn in impls.items():
        assert fn(s) == changes, (name, s)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp mô phỏng trạng thái độc lập")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp mô phỏng trạng thái độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên bỏ qua chữ hoa/chữ thường

In [21]:
def sai_hoa_thuong(s):
    count = 0
    for i in range(1, len(s)):
        if s[i] != s[i - 1]:                              # 'a' != 'A' là True
            count += 1
    return count


print("sai  :", sai_hoa_thuong("aAbBcC"), "| đúng:", Solution().countKeyChanges("aAbBcC"))

sai  : 5 | đúng: 2


**Vì sao sai:** `'a' != 'A'` trong Python, nên mỗi lần đổi Shift bị tính là đổi phím (`5` lần thay vì `2`). Phải chuẩn hóa bằng `lower()` (hoặc `upper()`) **cả hai** ký tự trước khi so sánh.

### Lỗi 2: Bắt đầu vòng lặp từ `0`

In [22]:
def sai_bat_dau_0(s):
    count = 0
    for i in range(len(s)):                                # i = 0: s[-1] là ký tự CUỐI
        if s[i].lower() != s[i - 1].lower():
            count += 1
    return count


print("sai  :", sai_bat_dau_0("ab"), "| đúng:", Solution().countKeyChanges("ab"))
print("sai  :", sai_bat_dau_0("abc"), "| đúng:", Solution().countKeyChanges("abc"))

sai  : 2 | đúng: 1
sai  : 3 | đúng: 2


**Vì sao sai:** ký tự đầu không có phím trước. Với `i = 0`, `s[i - 1] = s[-1]` là ký tự **cuối** chuỗi (Python cho phép chỉ số âm) nên ta so sánh đầu với cuối và đếm thêm (`"ab"` cho `2` thay vì `1`, `"abc"` cho `3` thay vì `2`). Hãy dùng `range(1, len(s))`.

### Lỗi 3: Đếm số nhóm thay vì số lần đổi

In [23]:
def sai_so_nhom(s):
    return sum(1 for _ in groupby(s.lower()))              # số nhóm, chưa trừ 1


print("sai  :", sai_so_nhom("aAbBcC"), "| đúng:", Solution().countKeyChanges("aAbBcC"))

sai  : 3 | đúng: 2


**Vì sao sai:** `3` nhóm (`aa`, `bb`, `cc`) chỉ có `2` chỗ nối. Đáp án là **số chỗ nối** `= số nhóm - 1`. Bài này giống bài "đếm hàng rào và cột": `n` đoạn cần `n - 1` ranh giới.

### Lỗi 4: Đếm số chữ cái khác nhau thay vì số lần đổi

In [24]:
def sai_set(s):
    return len(set(s.lower())) - 1


print("sai  :", sai_set("abab"), "| đúng:", Solution().countKeyChanges("abab"))

sai  : 1 | đúng: 3


**Vì sao sai:** đề đếm **số lần đổi phím**, nên quay lại phím cũ cũng tính. Chuỗi `"abab"` đổi phím `3` lần (`a→b`, `b→a`, `a→b`) nhưng chỉ có `2` chữ cái khác nhau. `set` làm mất thông tin về **thứ tự** và số lần lặp lại.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "đếm số lần **thay đổi** giữa các phần tử **liền kề**" → duyệt `i` từ `1`, so sánh `s[i]` với `s[i - 1]`; hoặc `zip(s, s[1:])`.
- "**Không phân biệt** hoa/thường" → chuẩn hóa **một lần** bằng `lower()` (hoặc `casefold()` cho chữ có dấu), rồi so sánh.
- Công thức nhớ: **số lần đổi = số nhóm liên tiếp − 1** (mẫu "rào và đoạn").
- Tuyệt đối tránh `s[i - 1]` khi `i = 0` (chỉ số âm lặng lẽ quấn vòng).
- Phân biệt: đếm **số giá trị khác nhau** (`set`) với đếm **số lần đổi** (liền kề).
- Mẹo bit `ord(c) & 31` hoặc `ord(c) | 32` để chuẩn hóa chữ cái ASCII, chỉ nên dùng khi biết chắc dữ liệu là chữ cái.

## 15. Biến thể của bài

1. Phiên bản **phân biệt** hoa/thường (mỗi chữ hoa là phím riêng).
2. Đếm số lần **dùng Shift** (chuyển giữa hoa và thường), kể cả khi cùng chữ cái.
3. Tìm **đoạn dài nhất** gõ liên tiếp cùng một phím.
4. Trả về **các vị trí `i`** mà ở đó có đổi phím.
5. **Nén chuỗi** kiểu run-length: `"aaBBBc"` thành `[("a", 2), ("b", 3), ("c", 1)]`.

Thử biến thể 2, 3 và 4:

In [25]:
def count_shift_changes(s):
    return sum(1 for i in range(1, len(s)) if s[i].isupper() != s[i - 1].isupper())


def longest_same_key_run(s):
    return max(len(list(g)) for _, g in groupby(s.lower()))


def change_positions(s):
    return [i for i in range(1, len(s)) if s[i].lower() != s[i - 1].lower()]


print("2.", count_shift_changes("aAbBcC"))                 # 5: hoa/thường đổi ở mọi cặp
print("3.", longest_same_key_run("aAAbbBBBcC"))            # 5: đoạn b, b, B, B, B
print("4.", change_positions("aAbBcC"))                     # [2, 4]
assert count_shift_changes("aAbBcC") == 5
assert longest_same_key_run("aAAbbBBBcC") == 5
assert change_positions("aAbBcC") == [2, 4]
assert len(change_positions("aAbBcC")) == Solution().countKeyChanges("aAbBcC")

2. 5
3. 5
4. [2, 4]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So sánh cặp, `lower()` hai phía | `s[i].lower() != s[i-1].lower()` | `O(n)` | `O(1)` | **cách nộp bài**, rõ ràng |
| 2. `lower` cả chuỗi + `zip` | `sum(a != b ...)` | `O(n)` | `O(n)` | code gọn |
| 3. `groupby` | số nhóm − 1 | `O(n)` | `O(1)` | khi cần thông tin từng nhóm |
| 4. `prev` + `ord & 31` | không tạo chuỗi mới | `O(n)` | `O(1)` | tiết kiệm bộ nhớ, chỉ cho chữ cái ASCII |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"Ab".lower()`, `list(zip("abc", "bc"))`, `'a' == 'A'`, `Solution().countKeyChanges("abcABC")`, `Solution().countKeyChanges("xXxXx")`.

**Bài 2 (tính tay).** Với `s = "aAbBBcAa"`: lập bảng từng cặp (có đổi phím hay không) và tính số lần đổi; đối chiếu bằng cách đếm số nhóm liên tiếp của `s.lower()` rồi trừ `1`.

**Bài 3 (điền TODO).** Hoàn thành `count_key_changes_todo(s)` theo Cách 1.

In [26]:
def count_key_changes_todo(s):
    count = 0
    # TODO 1: duyệt i từ 1 đến len(s) - 1
    #     TODO 2: nếu s[i].lower() khác s[i - 1].lower() thì count += 1
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `count_key_changes_groupby2(s)` bằng **`groupby`** (Cách 3) mà không nhìn code mẫu. Kiểm tra với năm ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_case_sensitive(s)`: chữ hoa và thường là **hai phím khác nhau**. (b) Viết `rle(s)`: nén chuỗi (đã chuẩn hóa chữ thường) thành danh sách `(chữ, số lần liên tiếp)`. (c) Viết `min_key_changes_after_one_edit(s)`: được **đổi đúng một ký tự** của `s` thành chữ cái bất kỳ; trả về **số lần đổi phím nhỏ nhất** có thể (không được bỏ qua việc đổi).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"ab"`, `[('a', 'b'), ('b', 'c')]`, `False`, `5` (`abcabc` đổi phím ở cả `5` cặp), `0`. Bài 3: vòng `for i in range(1, len(s))` rồi `if s[i].lower() != s[i - 1].lower(): count += 1`.
- **Mức 2 (vừa):** Bài 2: các cặp (`a,A`: không), (`A,b`: có), (`b,B`: không), (`B,B`: không), (`B,c`: có), (`c,A`: có), (`A,a`: không) nên `3`. Nhóm của `"aabbbcaa"`: `aa`, `bbb`, `c`, `aa` = `4` nhóm, `4 - 1 = 3`. Bài 4: `sum(1 for _ in groupby(s.lower())) - 1`.
- **Mức 3 (sâu):** Bài 5(a): so sánh `s[i] != s[i - 1]` trực tiếp. 5(b): `[(k, len(list(g))) for k, g in groupby(s.lower())]`. 5(c): thử mọi vị trí và mọi chữ cái thay thế (52 lựa chọn gồm hoa và thường) rồi lấy `min` của `countKeyChanges`; gợi ý: một thay đổi tại vị trí `i` chỉ ảnh hưởng hai cặp `(i - 1, i)` và `(i, i + 1)` nên có thể tính cục bộ; nhưng thử toàn bộ cho dễ vì `n <= 100`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert "Ab".lower() == "ab" and list(zip("abc", "bc")) == [("a", "b"), ("b", "c")]
assert ("a" == "A") is False
assert Solution().countKeyChanges("abcABC") == 5      # a b c a b c: mọi cặp đổi phím
assert Solution().countKeyChanges("xXxXx") == 0

# Bài 2
assert Solution().countKeyChanges("aAbBBcAa") == 3
assert len(list(groupby("aAbBBcAa".lower()))) - 1 == 3

# Bài 3
def count_key_changes_todo_ans(s):
    count = 0
    for i in range(1, len(s)):
        if s[i].lower() != s[i - 1].lower():
            count += 1
    return count

for s, expected in examples:
    assert count_key_changes_todo_ans(s) == expected

# Bài 4
def count_key_changes_groupby2(s):
    return sum(1 for _ in groupby(s.lower())) - 1

for s, expected in examples:
    assert count_key_changes_groupby2(s) == expected

# Bài 5(a)
def count_case_sensitive(s):
    return sum(1 for i in range(1, len(s)) if s[i] != s[i - 1])

assert count_case_sensitive("aAbBcC") == 5
assert count_case_sensitive("aaAA") == 1

# Bài 5(b)
def rle(s):
    return [(k, len(list(g))) for k, g in groupby(s.lower())]

assert rle("aaBBBc") == [("a", 2), ("b", 3), ("c", 1)]
assert rle("aAbBcC") == [("a", 2), ("b", 2), ("c", 2)]

# Bài 5(c)
def min_key_changes_after_one_edit(s):
    best = None
    for i in range(len(s)):
        for ch in LETTERS + LETTERS.upper():
            if ch == s[i]:
                continue                                    # bắt buộc phải đổi thành ký tự khác
            t = s[:i] + ch + s[i + 1:]
            c = Solution().countKeyChanges(t)
            if best is None or c < best:
                best = c
    return best

assert min_key_changes_after_one_edit("aab") == 0              # đổi 'b' thành 'a'
assert min_key_changes_after_one_edit("abc") == 1              # đổi giữa thành 'a' hoặc 'c': a a c -> 1
assert min_key_changes_after_one_edit("aaa") == 0              # đổi một 'a' thành 'A': vẫn cùng phím
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (so sánh cặp liền kề sau khi chuẩn hóa hoa/thường)

```python
class Solution:
    def countKeyChanges(self, s: str) -> int:
        count = 0
        for i in range(1, len(s)):
            if s[i].lower() != s[i - 1].lower():
                count += 1
        return count
```

**Checklist**

- [ ] Đã chuẩn hóa hoa/thường **cả hai** ký tự trước khi so sánh (`lower()` cả hai phía)?
- [ ] Vòng lặp bắt đầu từ `i = 1` (không so `s[0]` với `s[-1]`)?
- [ ] Đếm **số chỗ đổi**, không phải số nhóm (đã trừ `1`) hay số chữ cái khác nhau?
- [ ] Hiểu rằng quay lại phím cũ (`abab`) vẫn là đổi phím?
- [ ] Đã thử: một ký tự, toàn chữ giống nhau, hoa/thường xen kẽ cùng chữ, xen kẽ hai chữ, `n = 100`?
- [ ] Skeleton đúng `class Solution` / `def countKeyChanges(self, s: str) -> int`?